In [926]:
# load libraries 
import pandas as pd
import numpy as np
import re
import math
import plotly.graph_objects as go
import plotly.express as px
from pathlib import Path

In [927]:
# get the SUN covid study dataset
Covid51countries = pd.read_csv(Path("~/Projects/hypocognition/data/raw/Covid51countries.csv").expanduser())
Covid51countries

,country,countryname,StartDate,EndDate,Status,Progress,Duration,Finished,RecordedDate,home,...,gender,ladder,employment,losejob,covidsymptom,language,datasource,ISO3,longstring,na_count
0,10,Australia,04/05/2020 19:34,04/05/2020 19:40,0.0,100.0,337.0,1.0,04/05/2020 19:40,8.0,...,1.0,5.0,4,NaN,0.0,ZH-S,snowball,AUS,3,0
1,10,Australia,04/05/2020 20:26,04/05/2020 20:35,0.0,100.0,530.0,1.0,04/05/2020 20:35,9.0,...,2.0,7.0,3,NaN,NaN,ZH-S,snowball,AUS,3,0
2,10,Australia,05/05/2020 21:54,05/05/2020 22:07,0.0,100.0,793.0,1.0,05/05/2020 22:07,29.0,...,1.0,4.0,1,NaN,0.0,ZH-S,snowball,AUS,5,0
3,10,Australia,04/05/2020 20:29,04/05/2020 20:34,0.0,100.0,285.0,1.0,04/05/2020 20:34,73.0,...,2.0,5.0,4,NaN,0.0,ZH-S,snowball,AUS,4,0
4,10,Australia,19/04/2020 20:21,19/04/2020 20:27,0.0,100.0,367.0,1.0,19/04/2020 20:27,100.0,...,1.0,5.0,1,NaN,0.0,ZH-S,snowball,AUS,3,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24216,92,Kenya,25/04/2020 01:25,25/04/2020 02:44,0.0,100.0,4690.0,1.0,25/04/2020 02:44,100.0,...,2.0,4.0,4,NaN,0.0,EN,snowball,KEN,4,0
24217,92,Kenya,24/04/2020 11:06,24/04/2020 11:15,0.0,100.0,556.0,1.0,24/04/2020 11:15,100.0,...,1.0,6.0,5,0.0,0.0,EN,snowball,KEN,2,0
24218,92,Kenya,26/04/2020 10:16,26/04/2020 10:36,0.0,100.0,1218.0,1.0,26/04/2020 10:36,80.0,...,1.0,3.0,4,NaN,0.0,EN,snowball,KEN,3,0
24219,92,Kenya,23/04/2020 03:46,23/04/2020 04:13,0.0,100.0,1641.0,1.0,23/04/2020 04:13,3.0,...,1.0,3.0,4,NaN,0.0,EN,snowball,KEN,3,0


In [928]:
Covid51countries.columns

Index(['country', 'countryname', 'StartDate', 'EndDate', 'Status', 'Progress',
       'Duration', 'Finished', 'RecordedDate', 'home', 'gathering', 'distance',
       'hands', 'help_covid', 'donate_covid', 'volunteer_covid', 'help',
       'donate', 'volunteer', 'admiration', 'calm', 'compassion',
       'determination', 'moved', 'gratitude', 'hope', 'love', 'relief',
       'pleasure', 'anger', 'anxiety', 'boredom', 'confusion', 'disgust',
       'fear', 'frustration', 'loneliness', 'regret', 'sadness',
       'ERQ1_rumination', 'ERQ2_reappraisal', 'ERQ3_suppression',
       'ERQ4_socialsharing', 'ERQ5_distraction', 'ERQ6_acceptance', 'support',
       'connected', 'phy_healthy', 'mentally_healthy', 'stressed', 'tired',
       'depressed', 'res_1', 'res_2', 'euda_1', 'euda_2', 'swl', 'sympathy',
       'concerned', 'overwhelmed', 'distressed', 'self_vulnerable',
       'country_vulnerable', 'age', 'education', 'gender', 'ladder',
       'employment', 'losejob', 'covidsymptom', 'languag

In [929]:
Covid51countries = Covid51countries[['country', 'countryname', 'language','ISO3', 'admiration', 'calm', 'compassion',
       'determination', 'moved', 'gratitude', 'hope', 'love', 'relief',
       'pleasure', 'anger', 'anxiety', 'boredom', 'confusion', 'disgust',
       'fear', 'frustration', 'loneliness', 'regret', 'sadness']]

In [930]:
# add language names to the df
lang_names = pd.read_csv(Path("~/Projects/hypocognition/data/external/lang_names.csv"))
Covid51countries = Covid51countries.merge(lang_names, left_on="language", right_on="Code", how="left")
Covid51countries

,country,countryname,language,ISO3,admiration,calm,compassion,determination,moved,gratitude,...,boredom,confusion,disgust,fear,frustration,loneliness,regret,sadness,Code,Language_Name
0,10,Australia,ZH-S,AUS,5.0,3.0,3.0,5.0,4.0,4.0,...,2.0,2.0,2.0,1.0,2.0,1.0,1.0,1.0,ZH-S,Simplified Chinese
1,10,Australia,ZH-S,AUS,2.0,3.0,3.0,0.0,4.0,2.0,...,3.0,1.0,3.0,3.0,1.0,1.0,2.0,2.0,ZH-S,Simplified Chinese
2,10,Australia,ZH-S,AUS,3.0,3.0,0.0,0.0,0.0,3.0,...,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,ZH-S,Simplified Chinese
3,10,Australia,ZH-S,AUS,6.0,1.0,4.0,4.0,4.0,4.0,...,6.0,4.0,4.0,4.0,3.0,6.0,1.0,2.0,ZH-S,Simplified Chinese
4,10,Australia,ZH-S,AUS,0.0,6.0,6.0,6.0,2.0,0.0,...,4.0,5.0,2.0,4.0,2.0,1.0,0.0,3.0,ZH-S,Simplified Chinese
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24216,92,Kenya,EN,KEN,3.0,5.0,5.0,4.0,6.0,3.0,...,5.0,5.0,6.0,4.0,6.0,6.0,6.0,6.0,EN,English
24217,92,Kenya,EN,KEN,6.0,6.0,0.0,6.0,2.0,2.0,...,6.0,0.0,3.0,6.0,3.0,6.0,5.0,2.0,EN,English
24218,92,Kenya,EN,KEN,2.0,3.0,3.0,5.0,3.0,3.0,...,4.0,4.0,4.0,5.0,5.0,3.0,5.0,4.0,EN,English
24219,92,Kenya,EN,KEN,2.0,2.0,5.0,5.0,5.0,6.0,...,5.0,3.0,5.0,5.0,3.0,6.0,1.0,5.0,EN,English


In [931]:
# merge croatian, bosnian and serbian
scb = ["Serbian", "Croatian", "Bosnian"]
Covid51countries = Covid51countries.drop(columns=['language'])
Covid51countries.loc[Covid51countries["Language_Name"].isin(scb), "Language_Name"] = "Serbian-Croatian-Bosnian"


In [932]:
# We want to use on only one language of responses for each country
# Further, that language cannot be English

# Calculate the number of responses for each country
country_response_count = pd.DataFrame(Covid51countries[['countryname', 'ISO3']].value_counts())

#collect the languages which people used to respond from each country
languages_per_country = Covid51countries.groupby('countryname')['Language_Name'].value_counts().reset_index(name="count")

# mereg num resposes to languages people used
languages_per_country = languages_per_country.merge(
    country_response_count,
    on="countryname",
    how="left"
)

# Clean up
languages_per_country = languages_per_country.rename(columns={"count_x": "lang_responses", "count_y": "total_responses"})
languages_per_country["percent"] = languages_per_country["lang_responses"] / languages_per_country["total_responses"] * 100

languages_per_country

,countryname,Language_Name,lang_responses,total_responses,percent
0,Australia,Simplified Chinese,181,378,47.883598
1,Australia,English,156,378,41.269841
2,Australia,Indonesian,12,378,3.174603
3,Australia,Traditional Chinese,10,378,2.645503
4,Australia,Vietnamese,5,378,1.322751
...,...,...,...,...,...
491,Vietnam,Simplified Chinese,2,338,0.591716
492,Vietnam,French,1,338,0.295858
493,Vietnam,Japanese,1,338,0.295858
494,Vietnam,Polish,1,338,0.295858


In [933]:
non_eng = languages_per_country[languages_per_country['Language_Name'] != "English"]
non_eng

,countryname,Language_Name,lang_responses,total_responses,percent
0,Australia,Simplified Chinese,181,378,47.883598
2,Australia,Indonesian,12,378,3.174603
3,Australia,Traditional Chinese,10,378,2.645503
4,Australia,Vietnamese,5,378,1.322751
5,Australia,Danish,3,378,0.793651
...,...,...,...,...,...
491,Vietnam,Simplified Chinese,2,338,0.591716
492,Vietnam,French,1,338,0.295858
493,Vietnam,Japanese,1,338,0.295858
494,Vietnam,Polish,1,338,0.295858


In [934]:
non_eng[non_eng['Language_Name']=="Dari"]

,countryname,Language_Name,lang_responses,total_responses,percent
308,Netherlands,Dari,3,1348,0.222552
425,Ukraine,Dari,1,698,0.143266
454,United Kingdom (UK),Dari,1,662,0.151057


In [935]:
# gather list of top names
try:
    del top4
except:
    pass
for n in non_eng['countryname'].unique():
    df = non_eng[non_eng['countryname'] == n].reset_index(drop=True)
    try:
        top4 = pd.concat([top4, df.iloc[0:4]], ignore_index=True)
    except:
        top4 = df.iloc[0:4]

top4

,countryname,Language_Name,lang_responses,total_responses,percent
0,Australia,Simplified Chinese,181,378,47.883598
1,Australia,Indonesian,12,378,3.174603
2,Australia,Traditional Chinese,10,378,2.645503
3,Australia,Vietnamese,5,378,1.322751
4,Brazil,Brazilian Portuguese,306,325,94.153846
...,...,...,...,...,...
158,United States of America (USA),Japanese,13,952,1.365546
159,Vietnam,Vietnamese,292,338,86.390533
160,Vietnam,Danish,3,338,0.887574
161,Vietnam,Simplified Chinese,2,338,0.591716


In [936]:
# we are short the 2*50 rows we'd excpect.
top4['countryname'].value_counts()

countryname
Australia                         4
Brazil                            4
Canada                            4
China                             4
Denmark                           4
Hong Kong (S.A.R)                 4
Germany                           4
Greece                            4
France                            4
Finland                           4
Japan                             4
Malaysia                          4
Israel                            4
Italy                             4
Ireland                           4
Indonesia                         4
India                             4
Hungary                           4
United States of America (USA)    4
Vietnam                           4
Ukraine                           4
Taiwan                            4
Sweden                            4
United Kingdom (UK)               4
Turkey                            4
Russia                            4
Netherlands                       4
New Zealand     

In [937]:

covid_to_bila_nouns_full_lang_name_mapping = pd.read_csv(Path("~/Projects/hypocognition/data/external/covid_to_full_bila_lang_name_mapping.csv"))
covid_to_bila_nouns_full_lang_name_mapping

,code,study_language_names,bila_language_name_mapping,possible_alternative_bila_language_name_mappings
0,ZH-S,Simplified Chinese,Mandarin Chinese,"Beijing Mandarin, Wu Chinese"
1,EN,English,NaN,"Midland American English, Singlish, Devon, Sus..."
2,ID,Indonesian,Indonesian,"Standard Malay, Central Malay, Baba Malay"
3,ZH-T,Traditional Chinese,Mandarin Chinese,"Yue Chinese, Min Dong Chinese, Min Nan Chinese..."
4,VI,Vietnamese,Vietnamese,Nung (Viet Nam)
5,DA,Danish,Danish,NaN
6,AR,Arabic,Arabic,"Egyptian Arabic, Levantine Arabic, Moroccan Ar..."
7,ES,Spanish,Latin American Spanish,"Spanish, Mexican Spanish"
8,AFRI,Afrikaans,Afrikaans,Dutch
9,ES-ES,Spanish (Spain),Spanish,NaN


In [938]:
covid_to_bila_nouns_full_lang_name_mapping['study_language_names'].nunique()

48

In [939]:
map = set(covid_to_bila_nouns_full_lang_name_mapping['bila_language_name_mapping'].unique())
orig = set(covid_to_bila_nouns_full_lang_name_mapping['study_language_names'].unique())


In [940]:
# Add the mapping to our selection table
selection = top4.merge(
    covid_to_bila_nouns_full_lang_name_mapping,
    left_on="Language_Name",
    right_on = "study_language_names",
    how = "left"
).drop(columns=["code", "study_language_names"])
selection


,countryname,Language_Name,lang_responses,total_responses,percent,bila_language_name_mapping,possible_alternative_bila_language_name_mappings
0,Australia,Simplified Chinese,181,378,47.883598,Mandarin Chinese,"Beijing Mandarin, Wu Chinese"
1,Australia,Indonesian,12,378,3.174603,Indonesian,"Standard Malay, Central Malay, Baba Malay"
2,Australia,Traditional Chinese,10,378,2.645503,Mandarin Chinese,"Yue Chinese, Min Dong Chinese, Min Nan Chinese..."
3,Australia,Vietnamese,5,378,1.322751,Vietnamese,Nung (Viet Nam)
4,Brazil,Brazilian Portuguese,306,325,94.153846,Brazilian Portuguese,Portuguese
...,...,...,...,...,...,...,...
158,United States of America (USA),Japanese,13,952,1.365546,Japanese,Middle Chinese
159,Vietnam,Vietnamese,292,338,86.390533,Vietnamese,Nung (Viet Nam)
160,Vietnam,Danish,3,338,0.887574,Danish,NaN
161,Vietnam,Simplified Chinese,2,338,0.591716,Mandarin Chinese,"Beijing Mandarin, Wu Chinese"


In [941]:
selection['Language_Name'].nunique()

42

In [942]:
selection['bila_language_name_mapping'].nunique()

37

In [943]:
map - set(selection['bila_language_name_mapping'].unique())


{'Estonian', 'Filipino', 'Norwegian', 'Slovak', 'Slovenian'}

In [944]:
orig - set(selection['Language_Name'].unique())

{'English', 'Estonian', 'Filipino', 'Norwegian', 'Slovak', 'Slovenian'}

In [945]:
# lets look at the results
with pd.option_context('display.max_rows', None):
    display(selection)

,countryname,Language_Name,lang_responses,total_responses,percent,bila_language_name_mapping,possible_alternative_bila_language_name_mappings
0,Australia,Simplified Chinese,181,378,47.883598,Mandarin Chinese,"Beijing Mandarin, Wu Chinese"
1,Australia,Indonesian,12,378,3.174603,Indonesian,"Standard Malay, Central Malay, Baba Malay"
2,Australia,Traditional Chinese,10,378,2.645503,Mandarin Chinese,"Yue Chinese, Min Dong Chinese, Min Nan Chinese..."
3,Australia,Vietnamese,5,378,1.322751,Vietnamese,Nung (Viet Nam)
4,Brazil,Brazilian Portuguese,306,325,94.153846,Brazilian Portuguese,Portuguese
5,Brazil,Spanish,7,325,2.153846,Latin American Spanish,"Spanish, Mexican Spanish"
6,Brazil,Danish,2,325,0.615385,Danish,NaN
7,Brazil,Portuguese,2,325,0.615385,Portuguese,Brazilian Portuguese
8,Bulgaria,Bulgarian,269,276,97.463768,Bulgarian,NaN
9,Bulgaria,Arabic,1,276,0.362319,Arabic,"Egyptian Arabic, Levantine Arabic, Moroccan Ar..."


In [946]:
selection['Selected'] = np.nan
selection

,countryname,Language_Name,lang_responses,total_responses,percent,bila_language_name_mapping,possible_alternative_bila_language_name_mappings,Selected
0,Australia,Simplified Chinese,181,378,47.883598,Mandarin Chinese,"Beijing Mandarin, Wu Chinese",NaN
1,Australia,Indonesian,12,378,3.174603,Indonesian,"Standard Malay, Central Malay, Baba Malay",NaN
2,Australia,Traditional Chinese,10,378,2.645503,Mandarin Chinese,"Yue Chinese, Min Dong Chinese, Min Nan Chinese...",NaN
3,Australia,Vietnamese,5,378,1.322751,Vietnamese,Nung (Viet Nam),NaN
4,Brazil,Brazilian Portuguese,306,325,94.153846,Brazilian Portuguese,Portuguese,NaN
...,...,...,...,...,...,...,...,...
158,United States of America (USA),Japanese,13,952,1.365546,Japanese,Middle Chinese,NaN
159,Vietnam,Vietnamese,292,338,86.390533,Vietnamese,Nung (Viet Nam),NaN
160,Vietnam,Danish,3,338,0.887574,Danish,NaN,NaN
161,Vietnam,Simplified Chinese,2,338,0.591716,Mandarin Chinese,"Beijing Mandarin, Wu Chinese",NaN


In [947]:
# simplest way to get the table right is to just tweak it in excel and load it back in
selection.to_csv(Path("~/Projects/hypocognition/data/processed/outline_selection.csv").expanduser())


In [948]:
selection = pd.read_csv(Path("~/Projects/hypocognition/data/processed/selection_in_bila.csv").expanduser(), index_col=0).reset_index(drop=True)
selection = selection[selection['Selected']==1].rename(columns={'study_language_names':"Language_Name"} )

In [949]:
map - set(selection['bila_language_name_mapping'].unique())

{'Czech',
 'Estonian',
 'Filipino',
 'Hindi',
 'Kazakh',
 'Korean',
 'Malayalam',
 'Norwegian',
 'Polish',
 'Portuguese',
 'Romanian',
 'Slovak',
 'Slovenian'}

In [950]:
selection['bila_language_name_mapping'].nunique()

29

In [951]:
selection['countryname'].nunique()

39

In [952]:
selection['countryname'].unique()

array(['Brazil', 'Bulgaria', 'Chile', 'China', 'Colombia', 'Croatia',
       'Curacao', 'Denmark', 'Egypt', 'Finland', 'France', 'Georgia',
       'Germany', 'Greece', 'Hong Kong (S.A.R)', 'Hungary', 'Iceland',
       'India', 'Indonesia', 'Iran', 'Israel', 'Italy', 'Japan', 'Jordan',
       'Kazakhstan', 'Malaysia', 'Malta', 'Netherlands', 'Peru', 'Russia',
       'Serbia', 'South Africa', 'Spain', 'Sweden', 'Syria', 'Taiwan',
       'Turkey', 'Ukraine', 'Vietnam'], dtype=object)

In [953]:
orig - set(selection['Language_Name'].unique())

{'Catalan',
 'Czech',
 'Dari',
 'English',
 'Estonian',
 'Filipino',
 'Hindi',
 'Kazakh',
 'Korean',
 'Malayalam',
 'Norwegian',
 'Polish',
 'Portuguese',
 'Romanian',
 'Slovak',
 'Slovenian'}

In [954]:
# those are all good exclusions.
# Lets try filtering the dataset using our new selection
filtered = Covid51countries.merge(
    selection,
    on=['countryname', 'Language_Name'],
    how='inner'
)

filtered

,country,countryname,ISO3,admiration,calm,compassion,determination,moved,gratitude,hope,...,Code,Language_Name,language,bila_language_name_mapping,lang_responses,total_responses,percent,Selected,reason,possible_alternative_bila_language_name_mappings
0,109,Malaysia,MYS,1.0,6.0,1.0,0.0,0.0,0.0,6.0,...,MS,Malay,MS,Standard Malay,155,214,72.429907,1,Let's use Malay for Malaysia,"Central Malay, Baba Malay, Malayo"
1,109,Malaysia,MYS,4.0,4.0,2.0,3.0,2.0,3.0,4.0,...,MS,Malay,MS,Standard Malay,155,214,72.429907,1,Let's use Malay for Malaysia,"Central Malay, Baba Malay, Malayo"
2,109,Malaysia,MYS,3.0,3.0,3.0,3.0,3.0,1.0,0.0,...,MS,Malay,MS,Standard Malay,155,214,72.429907,1,Let's use Malay for Malaysia,"Central Malay, Baba Malay, Malayo"
3,109,Malaysia,MYS,6.0,3.0,6.0,4.0,6.0,5.0,6.0,...,MS,Malay,MS,Standard Malay,155,214,72.429907,1,Let's use Malay for Malaysia,"Central Malay, Baba Malay, Malayo"
4,109,Malaysia,MYS,5.0,5.0,5.0,6.0,4.0,6.0,5.0,...,MS,Malay,MS,Standard Malay,155,214,72.429907,1,Let's use Malay for Malaysia,"Central Malay, Baba Malay, Malayo"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15020,91,Kazakhstan,KAZ,1.0,3.0,3.0,1.0,5.0,5.0,3.0,...,RU,Russian,RU,Russian,279,291,95.876289,1,Let's use Russian for Kazakhstan,Belarusian
15021,91,Kazakhstan,KAZ,3.0,3.0,6.0,2.0,6.0,5.0,5.0,...,RU,Russian,RU,Russian,279,291,95.876289,1,Let's use Russian for Kazakhstan,Belarusian
15022,91,Kazakhstan,KAZ,3.0,3.0,3.0,3.0,4.0,4.0,0.0,...,RU,Russian,RU,Russian,279,291,95.876289,1,Let's use Russian for Kazakhstan,Belarusian
15023,91,Kazakhstan,KAZ,2.0,3.0,6.0,2.0,2.0,1.0,6.0,...,RU,Russian,RU,Russian,279,291,95.876289,1,Let's use Russian for Kazakhstan,Belarusian


In [955]:
set(selection['Language_Name'].unique())- set(filtered['Language_Name'].unique())

set()

In [956]:
set(selection['bila_language_name_mapping'].unique())- set(filtered['bila_language_name_mapping'].unique())

set()

In [957]:
selection['Language_Name'].nunique()

32

In [958]:
selection['countryname'].nunique()

39

In [959]:
# Check to make sure it worked
filtered.groupby('countryname')['Language_Name'].unique()

countryname
Brazil                       [Brazilian Portuguese]
Bulgaria                                [Bulgarian]
Chile                                     [Spanish]
China                          [Simplified Chinese]
Colombia                                  [Spanish]
Croatia                  [Serbian-Croatian-Bosnian]
Curacao                                     [Dutch]
Denmark                                    [Danish]
Egypt                                      [Arabic]
Finland                                   [Finnish]
France                                     [French]
Georgia                                  [Georgian]
Germany                                    [German]
Greece                                      [Greek]
Hong Kong (S.A.R)             [Traditional Chinese]
Hungary                                 [Hungarian]
Iceland                                 [Icelandic]
India                                     [Marathi]
Indonesia                              [Indonesian]


In [960]:
filtered.groupby('countryname')['bila_language_name_mapping'].unique()

countryname
Brazil                   [Brazilian Portuguese]
Bulgaria                            [Bulgarian]
Chile                  [Latin American Spanish]
China                        [Mandarin Chinese]
Colombia               [Latin American Spanish]
Croatia              [Serbian-Croatian-Bosnian]
Curacao                                 [Dutch]
Denmark                                [Danish]
Egypt                                  [Arabic]
Finland                               [Finnish]
France                                 [French]
Georgia                              [Georgian]
Germany                                [German]
Greece                                  [Greek]
Hong Kong (S.A.R)            [Mandarin Chinese]
Hungary                             [Hungarian]
Iceland                             [Icelandic]
India                                 [Marathi]
Indonesia                          [Indonesian]
Iran                                  [Persian]
Israel                      

Goals
* For each distinct language, what is the lexical ellaboration for each of the 20 emotions?
* For each distinct language, is there any statistical significance in the distribution of the results of any particular emotion?

In [961]:
emotions = ['admiration', 'calm', 'compassion',
       'determination', 'moved', 'gratitude', 'hope', 'love', 'relief',
       'pleasure', 'anger', 'anxiety', 'boredom', 'confusion', 'disgust',
       'fear', 'frustration', 'loneliness', 'regret', 'sadness']

In [962]:
# get the dataset of the dictionaries. We will look at how elaborated each of the twenty emotions are in each of the 39 languages
bila_nouns_full = pd.read_csv(Path("~/Projects/hypocognition/data/raw/bila_long_noun_lemmatized_full.csv").expanduser(), index_col=0)
bila_nouns_full

/tmp/ipykernel_2834355/268126284.py:2: DtypeWarning:

Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.



,id,word,nsenses,count,langname,glottocode,year,title,imprint,author,area,langfamily,affiliation,longitude,latitude,estimate,regression_elaboration,dictsize_data,simple_elaboration,log_count
0,chi.14718491,ability,2.0,2.0,Nancowry,nanc1247,1884.0,A dictionary of the Nancowry dialect of the Ni...,"Printed at the Home Dept. Press, 1884.",0,Eurasia,Austroasiatic,"Austroasiatic, Nicobaric, Nuclear Nicobaric, C...",93.3921,7.94812,-8.725128,-1.424169,10499,0.000190,1.098612
1,chi.14718491,accomplice,0.0,0.0,0,0,0.0,0,0,0,0,0,0,0.0000,0.00000,-9.823849,-0.156552,10499,0.000000,0.000000
2,chi.14718491,account,14.0,9.0,Nancowry,nanc1247,1884.0,A dictionary of the Nancowry dialect of the Ni...,"Printed at the Home Dept. Press, 1884.",0,Eurasia,Austroasiatic,"Austroasiatic, Nicobaric, Nuclear Nicobaric, C...",93.3921,7.94812,-7.520776,-2.250524,10499,0.000857,2.302585
3,chi.14718491,acre,0.0,0.0,0,0,0.0,0,0,0,0,0,0,0.0000,0.00000,-9.823849,-0.444142,10499,0.000000,0.000000
4,chi.14718491,act,15.0,6.0,Nancowry,nanc1247,1884.0,A dictionary of the Nancowry dialect of the Ni...,"Printed at the Home Dept. Press, 1884.",0,Eurasia,Austroasiatic,"Austroasiatic, Nicobaric, Nuclear Nicobaric, C...",93.3921,7.94812,0.000000,0.000000,10499,0.000571,1.945910
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5282195,wu.89119131142,prouds,0.0,0.0,0,0,0.0,0,0,0,0,0,0,0.0000,0.00000,0.000000,0.000000,33359,0.000000,0.000000
5282196,wu.89119131142,facilitator,0.0,0.0,0,0,0.0,0,0,0,0,0,0,0.0000,0.00000,0.000000,0.000000,33359,0.000000,0.000000
5282197,wu.89119131142,teacher-librarian,0.0,0.0,0,0,0.0,0,0,0,0,0,0,0.0000,0.00000,0.000000,0.000000,33359,0.000000,0.000000
5282198,wu.89119131142,pandani,0.0,0.0,0,0,0.0,0,0,0,0,0,0,0.0000,0.00000,0.000000,0.000000,33359,0.000000,0.000000


In [963]:
with open(Path("~/Projects/hypocognition/data/external/stopwords.txt").expanduser()) as f:
    words = [line.strip() for line in f if line.strip()]

In [964]:
# remove stop words
bila_nouns_full = bila_nouns_full[~bila_nouns_full["word"].isin(words)]
bila_nouns_full

,id,word,nsenses,count,langname,glottocode,year,title,imprint,author,area,langfamily,affiliation,longitude,latitude,estimate,regression_elaboration,dictsize_data,simple_elaboration,log_count
0,chi.14718491,ability,2.0,2.0,Nancowry,nanc1247,1884.0,A dictionary of the Nancowry dialect of the Ni...,"Printed at the Home Dept. Press, 1884.",0,Eurasia,Austroasiatic,"Austroasiatic, Nicobaric, Nuclear Nicobaric, C...",93.3921,7.94812,-8.725128,-1.424169,10499,0.000190,1.098612
1,chi.14718491,accomplice,0.0,0.0,0,0,0.0,0,0,0,0,0,0,0.0000,0.00000,-9.823849,-0.156552,10499,0.000000,0.000000
2,chi.14718491,account,14.0,9.0,Nancowry,nanc1247,1884.0,A dictionary of the Nancowry dialect of the Ni...,"Printed at the Home Dept. Press, 1884.",0,Eurasia,Austroasiatic,"Austroasiatic, Nicobaric, Nuclear Nicobaric, C...",93.3921,7.94812,-7.520776,-2.250524,10499,0.000857,2.302585
3,chi.14718491,acre,0.0,0.0,0,0,0.0,0,0,0,0,0,0,0.0000,0.00000,-9.823849,-0.444142,10499,0.000000,0.000000
6,chi.14718491,actor,0.0,0.0,0,0,0.0,0,0,0,0,0,0,0.0000,0.00000,-9.823849,-1.633890,10499,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5282195,wu.89119131142,prouds,0.0,0.0,0,0,0.0,0,0,0,0,0,0,0.0000,0.00000,0.000000,0.000000,33359,0.000000,0.000000
5282196,wu.89119131142,facilitator,0.0,0.0,0,0,0.0,0,0,0,0,0,0,0.0000,0.00000,0.000000,0.000000,33359,0.000000,0.000000
5282197,wu.89119131142,teacher-librarian,0.0,0.0,0,0,0.0,0,0,0,0,0,0,0.0000,0.00000,0.000000,0.000000,33359,0.000000,0.000000
5282198,wu.89119131142,pandani,0.0,0.0,0,0,0.0,0,0,0,0,0,0,0.0000,0.00000,0.000000,0.000000,33359,0.000000,0.000000


In [965]:
tot_words = bila_nouns_full.groupby('id')['word'].nunique().reset_index(name='Total_words_in_dict')
tot_counts = bila_nouns_full.groupby('id')['count'].sum().reset_index(name='Total_counts_in_dict')


In [966]:
# I think we will needs these later
dictionary_means = (
    bila_nouns_full
        .groupby('id', as_index=False)['count']
        .mean()
        .rename(columns={'count': 'dictionary_count_mean'})
)


In [967]:
# filter just the emotions
bila_nouns_full_emotions = bila_nouns_full[bila_nouns_full['word'].isin(emotions)].reset_index(drop=True)
bila_nouns_full_emotions

,id,word,nsenses,count,langname,glottocode,year,title,imprint,author,area,langfamily,affiliation,longitude,latitude,estimate,regression_elaboration,dictsize_data,simple_elaboration,log_count
0,chi.14718491,fear,8.0,9.0,Nancowry,nanc1247,1884.0,A dictionary of the Nancowry dialect of the Ni...,"Printed at the Home Dept. Press, 1884.",0,Eurasia,Austroasiatic,"Austroasiatic, Nicobaric, Nuclear Nicobaric, C...",93.3921,7.94812,-7.520776,-1.337584,10499,0.000857,2.302585
1,chi.14718491,pleasure,5.0,5.0,Nancowry,nanc1247,1884.0,A dictionary of the Nancowry dialect of the Ni...,"Printed at the Home Dept. Press, 1884.",0,Eurasia,Austroasiatic,"Austroasiatic, Nicobaric, Nuclear Nicobaric, C...",93.3921,7.94812,-8.031819,-1.571145,10499,0.000476,1.791759
2,chi.14718491,regret,5.0,2.0,Nancowry,nanc1247,1884.0,A dictionary of the Nancowry dialect of the Ni...,"Printed at the Home Dept. Press, 1884.",0,Eurasia,Austroasiatic,"Austroasiatic, Nicobaric, Nuclear Nicobaric, C...",93.3921,7.94812,-8.725128,-0.486657,10499,0.000190,1.098612
3,chi.14718491,admiration,0.0,0.0,0,0,0.0,0,0,0,0,0,0,0.0000,0.00000,-9.823849,-0.710507,10499,0.000000,0.000000
4,chi.14718491,anger,5.0,2.0,Nancowry,nanc1247,1884.0,A dictionary of the Nancowry dialect of the Ni...,"Printed at the Home Dept. Press, 1884.",0,Eurasia,Austroasiatic,"Austroasiatic, Nicobaric, Nuclear Nicobaric, C...",93.3921,7.94812,-8.725128,-2.335062,10499,0.000190,1.098612
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11083,wu.89119131142,disgust,3.0,16.0,Herero,here1253,1989.0,An English-Herero dictionary : with an introdu...,"J.C. Juta, 1883.",0,Africa,Atlantic-Congo,"Atlantic-Congo, Volta-Congo, Benue-Congo, Bant...",20.5655,-21.02310,-7.795743,3.869782,33359,0.000480,2.833213
11084,wu.89119131142,frustration,0.0,0.0,0,0,0.0,0,0,0,0,0,0,0.0000,0.00000,-10.629344,-0.596842,33359,0.000000,0.000000
11085,wu.89119131142,loneliness,3.0,4.0,Herero,here1253,1989.0,An English-Herero dictionary : with an introdu...,"J.C. Juta, 1883.",0,Africa,Atlantic-Congo,"Atlantic-Congo, Volta-Congo, Benue-Congo, Bant...",20.5655,-21.02310,-9.019809,1.825273,33359,0.000120,1.609438
11086,wu.89119131142,relief,11.0,10.0,Herero,here1253,1989.0,An English-Herero dictionary : with an introdu...,"J.C. Juta, 1883.",0,Africa,Atlantic-Congo,"Atlantic-Congo, Volta-Congo, Benue-Congo, Bant...",20.5655,-21.02310,-8.231207,0.866433,33359,0.000300,2.397895


the dataset is missing two of the survey emotions, {'calm', 'moved'}

In [968]:
filtered.columns

Index(['country', 'countryname', 'ISO3', 'admiration', 'calm', 'compassion',
       'determination', 'moved', 'gratitude', 'hope', 'love', 'relief',
       'pleasure', 'anger', 'anxiety', 'boredom', 'confusion', 'disgust',
       'fear', 'frustration', 'loneliness', 'regret', 'sadness', 'Code',
       'Language_Name', 'language', 'bila_language_name_mapping',
       'lang_responses', 'total_responses', 'percent', 'Selected', 'reason',
       'possible_alternative_bila_language_name_mappings'],
      dtype='object')

In [969]:
filtered['bila_language_name_mapping'].nunique()

29

In [970]:
filtered['Language_Name'].nunique()

32

In [971]:
filtered['countryname'].nunique()

39

In [972]:
# already filtered out everything but the 18 emotions
# now we want to filter evrything but the 29 dictionaries
# Where are lang_name and Language_Names the Same?
bila_nouns_full_emotions_filtered = bila_nouns_full_emotions[bila_nouns_full_emotions['langname'].isin(filtered['bila_language_name_mapping'].unique())].reset_index(drop=True)
bila_nouns_full_emotions_filtered

,id,word,nsenses,count,langname,glottocode,year,title,imprint,author,area,langfamily,affiliation,longitude,latitude,estimate,regression_elaboration,dictsize_data,simple_elaboration,log_count
0,coo.31924067983704,fear,8.0,126.0,Turkish,nucl1301,1991.0,Redhouse yeni Türkçe-İngilizce sözlük = New Re...,"Redhouse Yayınevi, 1991, c1968.",0,Eurasia,Turkic,"Turkic, Common Turkic, Oghuz, Nuclear Oghuz, W...",32.8667,39.8667,-6.924086,1.952545,121674,0.001036,4.844187
1,coo.31924067983704,pleasure,5.0,107.0,Turkish,nucl1301,1991.0,Redhouse yeni Türkçe-İngilizce sözlük = New Re...,"Redhouse Yayınevi, 1991, c1968.",0,Eurasia,Turkic,"Turkic, Common Turkic, Oghuz, Nuclear Oghuz, W...",32.8667,39.8667,-7.086289,3.153327,121674,0.000879,4.682131
2,coo.31924067983704,regret,5.0,41.0,Turkish,nucl1301,1991.0,Redhouse yeni Türkçe-İngilizce sözlük = New Re...,"Redhouse Yayınevi, 1991, c1968.",0,Eurasia,Turkic,"Turkic, Common Turkic, Oghuz, Nuclear Oghuz, W...",32.8667,39.8667,-8.031261,2.670962,121674,0.000337,3.737670
3,coo.31924067983704,admiration,3.0,10.0,Turkish,nucl1301,1991.0,Redhouse yeni Türkçe-İngilizce sözlük = New Re...,"Redhouse Yayınevi, 1991, c1968.",0,Eurasia,Turkic,"Turkic, Common Turkic, Oghuz, Nuclear Oghuz, W...",32.8667,39.8667,-9.371276,-0.854986,121674,0.000082,2.397895
4,coo.31924067983704,anger,5.0,85.0,Turkish,nucl1301,1991.0,Redhouse yeni Türkçe-İngilizce sözlük = New Re...,"Redhouse Yayınevi, 1991, c1968.",0,Eurasia,Turkic,"Turkic, Common Turkic, Oghuz, Nuclear Oghuz, W...",32.8667,39.8667,-7.314243,0.579516,121674,0.000699,4.454347
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
499,uva.x004877953,disgust,3.0,13.0,Japanese,nucl1643,1974.0,Kenkyusha's new Japanese-English dictionary. K...,Kenkyusha [1974],0,Eurasia,Japonic,"Japonic, Japanesic, Japan-Taiwan Japanese",135.0000,35.0000,-9.940215,-4.505457,282895,0.000046,2.639057
500,uva.x004877953,frustration,3.0,6.0,Japanese,nucl1643,1974.0,Kenkyusha's new Japanese-English dictionary. K...,Kenkyusha [1974],0,Eurasia,Japonic,"Japonic, Japanesic, Japan-Taiwan Japanese",135.0000,35.0000,-10.633387,-1.588076,282895,0.000021,1.945910
501,uva.x004877953,loneliness,3.0,7.0,Japanese,nucl1643,1974.0,Kenkyusha's new Japanese-English dictionary. K...,Kenkyusha [1974],0,Eurasia,Japonic,"Japonic, Japanesic, Japan-Taiwan Japanese",135.0000,35.0000,-10.499852,-1.873436,282895,0.000025,2.079442
502,uva.x004877953,relief,11.0,89.0,Japanese,nucl1643,1974.0,Kenkyusha's new Japanese-English dictionary. K...,Kenkyusha [1974],0,Eurasia,Japonic,"Japonic, Japanesic, Japan-Taiwan Japanese",135.0000,35.0000,-8.079201,3.907437,282895,0.000315,4.499810


In [973]:
print(bila_nouns_full_emotions_filtered['langname'].nunique())

29


In [974]:
filtered = filtered.loc[:, ~filtered.columns.duplicated()]

In [975]:

bila_nouns_full_emotions_filtered['langname'].nunique()

29

In [976]:
word_counts = (
    bila_nouns_full_emotions_filtered
    .groupby("langname")["word"]
    .nunique()
    .reset_index(name="n_unique_words")
)
word_counts

,langname,n_unique_words
0,Afrikaans,18
1,Arabic,18
2,Brazilian Portuguese,8
3,Bulgarian,17
4,Danish,18
5,Dutch,18
6,Finnish,18
7,French,18
8,Georgian,16
9,German,18


In [977]:
bila_nouns_full_emotions_filtered.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 504 entries, 0 to 503
Data columns (total 20 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   id                      504 non-null    object 
 1   word                    504 non-null    object 
 2   nsenses                 504 non-null    float64
 3   count                   504 non-null    float64
 4   langname                504 non-null    object 
 5   glottocode              504 non-null    object 
 6   year                    504 non-null    float64
 7   title                   504 non-null    object 
 8   imprint                 504 non-null    object 
 9   author                  504 non-null    object 
 10  area                    504 non-null    object 
 11  langfamily              504 non-null    object 
 12  affiliation             504 non-null    object 
 13  longitude               504 non-null    float64
 14  latitude                504 non-null    fl

In [978]:
# add the rows of the emotions words which don't appear in each dictionary

full_index = pd.MultiIndex.from_product(
    [bila_nouns_full_emotions_filtered["id"].unique(), bila_nouns_full_emotions_filtered["word"].unique()],
    names=["id", "word"]
)


In [979]:
# use the full index to add the missing values
bila_nouns_full_emotions_filtered_full = (
    bila_nouns_full_emotions_filtered
    .set_index(["id", "word"])
    .reindex(full_index)
    .reset_index()
)
num_cols = ["nsenses", "count", "log_count", 'regression_elaboration', 'dictsize_data', 'simple_elaboration']
bila_nouns_full_emotions_filtered_full.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 522 entries, 0 to 521
Data columns (total 20 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   id                      522 non-null    object 
 1   word                    522 non-null    object 
 2   nsenses                 504 non-null    float64
 3   count                   504 non-null    float64
 4   langname                504 non-null    object 
 5   glottocode              504 non-null    object 
 6   year                    504 non-null    float64
 7   title                   504 non-null    object 
 8   imprint                 504 non-null    object 
 9   author                  504 non-null    object 
 10  area                    504 non-null    object 
 11  langfamily              504 non-null    object 
 12  affiliation             504 non-null    object 
 13  longitude               504 non-null    float64
 14  latitude                504 non-null    fl

In [980]:

#set the number columns to 0 where NaN
bila_nouns_full_emotions_filtered_full[num_cols] = bila_nouns_full_emotions_filtered_full[num_cols].fillna(0)
meta_cols = [
    "langname", "glottocode", "year", "title", "imprint", "author",
    "area", "langfamily", "affiliation", "longitude", "latitude"
]
bila_nouns_full_emotions_filtered_full.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 522 entries, 0 to 521
Data columns (total 20 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   id                      522 non-null    object 
 1   word                    522 non-null    object 
 2   nsenses                 522 non-null    float64
 3   count                   522 non-null    float64
 4   langname                504 non-null    object 
 5   glottocode              504 non-null    object 
 6   year                    504 non-null    float64
 7   title                   504 non-null    object 
 8   imprint                 504 non-null    object 
 9   author                  504 non-null    object 
 10  area                    504 non-null    object 
 11  langfamily              504 non-null    object 
 12  affiliation             504 non-null    object 
 13  longitude               504 non-null    float64
 14  latitude                504 non-null    fl

In [981]:
# fill the those zero rows with dictionary meta -data
bila_nouns_full_emotions_filtered_full[meta_cols] = (
    bila_nouns_full_emotions_filtered_full
    .groupby("id")[meta_cols]
    .transform("first")
)

bila_nouns_full_emotions_filtered_full.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 522 entries, 0 to 521
Data columns (total 20 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   id                      522 non-null    object 
 1   word                    522 non-null    object 
 2   nsenses                 522 non-null    float64
 3   count                   522 non-null    float64
 4   langname                522 non-null    object 
 5   glottocode              522 non-null    object 
 6   year                    522 non-null    float64
 7   title                   522 non-null    object 
 8   imprint                 522 non-null    object 
 9   author                  522 non-null    object 
 10  area                    522 non-null    object 
 11  langfamily              522 non-null    object 
 12  affiliation             522 non-null    object 
 13  longitude               522 non-null    float64
 14  latitude                522 non-null    fl

In [982]:
# now we're don emaking datasets, this is a table of just the stuff we need for correlations
elab_per_emotion = bila_nouns_full_emotions_filtered_full[['langname',   'glottocode','word', 'count',  'id', 'year',"simple_elaboration", "regression_elaboration",	"dictsize_data"	]]
elab_per_emotion

,langname,glottocode,word,count,id,year,simple_elaboration,regression_elaboration,dictsize_data
0,Turkish,nucl1301,fear,126.0,coo.31924067983704,1991.0,0.001036,1.952545,121674.0
1,Turkish,nucl1301,pleasure,107.0,coo.31924067983704,1991.0,0.000879,3.153327,121674.0
2,Turkish,nucl1301,regret,41.0,coo.31924067983704,1991.0,0.000337,2.670962,121674.0
3,Turkish,nucl1301,admiration,10.0,coo.31924067983704,1991.0,0.000082,-0.854986,121674.0
4,Turkish,nucl1301,anger,85.0,coo.31924067983704,1991.0,0.000699,0.579516,121674.0
...,...,...,...,...,...,...,...,...,...
517,Japanese,nucl1643,disgust,13.0,uva.x004877953,1974.0,0.000046,-4.505457,282895.0
518,Japanese,nucl1643,frustration,6.0,uva.x004877953,1974.0,0.000021,-1.588076,282895.0
519,Japanese,nucl1643,loneliness,7.0,uva.x004877953,1974.0,0.000025,-1.873436,282895.0
520,Japanese,nucl1643,relief,89.0,uva.x004877953,1974.0,0.000315,3.907437,282895.0


In [983]:
elab_per_emotion.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 522 entries, 0 to 521
Data columns (total 9 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   langname                522 non-null    object 
 1   glottocode              522 non-null    object 
 2   word                    522 non-null    object 
 3   count                   522 non-null    float64
 4   id                      522 non-null    object 
 5   year                    522 non-null    float64
 6   simple_elaboration      522 non-null    float64
 7   regression_elaboration  522 non-null    float64
 8   dictsize_data           522 non-null    float64
dtypes: float64(5), object(4)
memory usage: 36.8+ KB


In [984]:
# Make table of the 18 languages for each country
stats_by_country = (
    filtered
    .groupby(['countryname', 'bila_language_name_mapping'])[emotions]
    .agg(['mean', 'std'])
)
stats_by_country = pd.DataFrame(stats_by_country)
stats_by_country

admiration                calm  \
                                                   mean       std      mean   
countryname       bila_language_name_mapping                                  
Brazil            Brazilian Portuguese         3.540984  1.745047  3.421569   
Bulgaria          Bulgarian                    2.498141  1.980548  3.334572   
Chile             Latin American Spanish       2.867521  1.901080  2.899358   
China             Mandarin Chinese             4.365940  1.736698  3.914807   
Colombia          Latin American Spanish       3.409836  1.974022  3.418033   
Croatia           Serbian-Croatian-Bosnian     2.548571  1.680610  3.336502   
Curacao           Dutch                        3.627778  1.743401  3.961111   
Denmark           Danish                       2.668919  1.513605  3.445946   
Egypt             Arabic                       2.547472  1.886575  3.622222   
Finland           Finnish                      2.738462  1.547467  3.726923   
France            French                       3.357977  1.801764  3.746094   
Georgia           Georgian                     2.523605  1.866354  3.248927   
Germany           German                       2.804124  1.800225  3.628866   
Greece            Greek                        2.328889  1.887059  3.337778   
Hong Kong (S.A.R) Mandarin Chinese             2.669065  1.770894  3.064748   
Hungary           Hungarian                    2.679688  1.925815  3.265625   
Iceland           Icelandic                    3.469741  1.867297  3.299712   
India             Marathi                      3.500000  1.768410  3.982143   
Indonesia         Indonesian                   3.201863  1.577327  3.834365   
Iran              Persian                      2.347826  1.745782  2.774194   
Israel            Hebrew                       1.569536  1.718565  3.240000   
Italy             Italian                      2.966981  1.900316  3.327059   
Japan             Japanese                     2.602176  1.523357  3.559441   
Jordan            Arabic                       3.108225  1.965235  3.952586   
Kazakhstan        Russian                      2.438849  1.958468  3.469534   
Malaysia          Standard Malay               3.793548  1.548761  4.000000   
Netherlands       Dutch                        3.325048  1.646279  3.671756   
Peru              Spanish                      3.706161  1.884501  3.693023   
Russia            Russian                      2.553043  1.774840  3.349565   
Serbia            Serbian-Croatian-Bosnian     2.716332  1.845246  3.518519   
South Africa      Afrikaans                    2.946429  1.712904  3.642857   
Spain             Spanish                      3.428571  1.994498  3.428571   
Sweden            Swedish                      3.300000  1.774224  3.580000   
Syria             Arabic                       1.849372  1.996402  3.377593   
Taiwan            Mandarin Chinese             3.993307  1.660520  3.666667   
Turkey            Turkish                      2.027888  1.760460  3.611111   
Ukraine           Ukrainian                    2.856489  1.644428  3.203957   
Vietnam           Vietnamese                   4.219178  1.746841  4.263699   

                                                       compassion            \
                                                   std       mean       std   
countryname       bila_language_name_mapping                                  
Brazil            Brazilian Portuguese        1.502585   4.633987  1.319471   
Bulgaria          Bulgarian                   1.712431   4.313433  1.590788   
Chile             Latin American Spanish      1.546330   3.893390  1.693798   
China             Mandarin Chinese            1.683731   4.240976  1.701800   
Colombia          Latin American Spanish      1.419115   4.024590  1.678563   
Croatia           Serbian-Croatian-Bosnian    1.487369   2.916190  1.657917   
Curacao           Dutch                       1.607816   4.777778  1.174951   
Denmark        

In [985]:
# lets just flatten out the 3 level column names
stats_by_country.columns = [
    f"{emotion}_{stat}" for emotion, stat in stats_by_country.columns
]
stats_by_country = stats_by_country.reset_index()
stats_by_country

,countryname,bila_language_name_mapping,admiration_mean,admiration_std,calm_mean,calm_std,compassion_mean,compassion_std,determination_mean,determination_std,...,fear_mean,fear_std,frustration_mean,frustration_std,loneliness_mean,loneliness_std,regret_mean,regret_std,sadness_mean,sadness_std
0,Brazil,Brazilian Portuguese,3.540984,1.745047,3.421569,1.502585,4.633987,1.319471,3.725490,1.593986,...,3.603279,1.757500,3.437908,1.941201,2.518033,2.069635,1.934426,1.881951,3.321311,1.833991
1,Bulgaria,Bulgarian,2.498141,1.980548,3.334572,1.712431,4.313433,1.590788,3.565056,1.684113,...,2.561338,1.954938,3.029740,2.089207,2.334572,2.154385,2.929104,2.009018,3.252788,1.930519
2,Chile,Latin American Spanish,2.867521,1.901080,2.899358,1.546330,3.893390,1.693798,3.370450,1.700109,...,3.528785,1.836746,4.119914,1.731608,2.869658,2.067930,2.352564,1.969156,3.799145,1.745670
3,China,Mandarin Chinese,4.365940,1.736698,3.914807,1.683731,4.240976,1.701800,4.165992,1.693406,...,1.809500,1.726083,1.815642,1.763461,1.812437,1.826492,1.477009,1.701083,2.256709,1.848471
4,Colombia,Latin American Spanish,3.409836,1.974022,3.418033,1.419115,4.024590,1.678563,3.818182,1.653280,...,3.024590,1.943258,3.409836,1.897167,2.578512,2.170837,1.721311,1.601973,3.459016,1.916210
5,Croatia,Serbian-Croatian-Bosnian,2.548571,1.680610,3.336502,1.487369,2.916190,1.657917,3.516190,1.441686,...,2.740952,1.825796,3.420952,1.812788,2.682510,1.955690,2.798479,1.710828,2.990494,1.760385
6,Curacao,Dutch,3.627778,1.743401,3.961111,1.607816,4.777778,1.174951,4.000000,1.549914,...,2.477778,1.933115,3.005556,1.877527,1.894444,1.853570,1.350000,1.645970,2.622222,1.906049
7,Denmark,Danish,2.668919,1.513605,3.445946,1.531012,4.067568,1.450602,3.351351,1.483996,...,1.824324,1.610832,3.148649,1.852980,2.277027,2.026427,1.479730,1.655622,1.601351,1.732993
8,Egypt,Arabic,2.547472,1.886575,3.622222,1.728264,4.173006,1.710858,3.161090,1.728845,...,3.556790,1.953694,3.419434,2.007981,3.686275,2.140949,2.710591,2.129295,4.047970,1.806232
9,Finland,Finnish,2.738462,1.547467,3.726923,1.380520,4.042471,1.350471,3.532819,1.327221,...,2.509653,1.600298,3.846154,1.699246,2.776062,1.927978,1.534615,1.570343,2.953668,1.672212


In [986]:
map - set(stats_by_country['bila_language_name_mapping'].unique())

{'Czech',
 'Estonian',
 'Filipino',
 'Hindi',
 'Kazakh',
 'Korean',
 'Malayalam',
 'Norwegian',
 'Polish',
 'Portuguese',
 'Romanian',
 'Slovak',
 'Slovenian',
 nan}

In [987]:
# it was very wide, leyts make it narrow, longer, and more robust
stats_long = (
    stats_by_country
    .set_index(['countryname', 'bila_language_name_mapping'])
    .filter(regex='_(mean|std)$')
    .stack()
    .reset_index()
)

stats_long[['word', 'stat']] = stats_long['level_2'].str.rsplit('_', n=1, expand=True)
stats_long = stats_long.rename(columns={0: 'value'}).drop(columns='level_2')

stats_long = (
    stats_long
    .pivot_table(
        index=['countryname', 'bila_language_name_mapping', 'word'],
        columns='stat',
        values='value'
    )
    .reset_index()
)


In [988]:
#Merge our cleaned survey data with the BILA dictionary data
merged = elab_per_emotion.merge(
    stats_long,
    left_on=['langname', 'word'],
    right_on=['bila_language_name_mapping', 'word'],
    how='left'
)

merged

,langname,glottocode,word,count,id,year,simple_elaboration,regression_elaboration,dictsize_data,countryname,bila_language_name_mapping,mean,std
0,Turkish,nucl1301,fear,126.0,coo.31924067983704,1991.0,0.001036,1.952545,121674.0,Turkey,Turkish,2.948617,1.892269
1,Turkish,nucl1301,pleasure,107.0,coo.31924067983704,1991.0,0.000879,3.153327,121674.0,Turkey,Turkish,3.083665,1.643463
2,Turkish,nucl1301,regret,41.0,coo.31924067983704,1991.0,0.000337,2.670962,121674.0,Turkey,Turkish,1.768000,1.788567
3,Turkish,nucl1301,admiration,10.0,coo.31924067983704,1991.0,0.000082,-0.854986,121674.0,Turkey,Turkish,2.027888,1.760460
4,Turkish,nucl1301,anger,85.0,coo.31924067983704,1991.0,0.000699,0.579516,121674.0,Turkey,Turkish,3.460000,1.871473
...,...,...,...,...,...,...,...,...,...,...,...,...,...
679,Japanese,nucl1643,disgust,13.0,uva.x004877953,1974.0,0.000046,-4.505457,282895.0,Japan,Japanese,2.854701,1.747655
680,Japanese,nucl1643,frustration,6.0,uva.x004877953,1974.0,0.000021,-1.588076,282895.0,Japan,Japanese,3.111888,1.729330
681,Japanese,nucl1643,loneliness,7.0,uva.x004877953,1974.0,0.000025,-1.873436,282895.0,Japan,Japanese,2.007764,1.817621
682,Japanese,nucl1643,relief,89.0,uva.x004877953,1974.0,0.000315,3.907437,282895.0,Japan,Japanese,2.403263,1.417428


In [989]:
merged.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 684 entries, 0 to 683
Data columns (total 13 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   langname                    684 non-null    object 
 1   glottocode                  684 non-null    object 
 2   word                        684 non-null    object 
 3   count                       684 non-null    float64
 4   id                          684 non-null    object 
 5   year                        684 non-null    float64
 6   simple_elaboration          684 non-null    float64
 7   regression_elaboration      684 non-null    float64
 8   dictsize_data               684 non-null    float64
 9   countryname                 684 non-null    object 
 10  bila_language_name_mapping  684 non-null    object 
 11  mean                        684 non-null    float64
 12  std                         684 non-null    float64
dtypes: float64(7), object(6)
memory usa

In [990]:
# add the dictionary_count_mean column
merged = merged.merge(
    dictionary_means,
    left_on='id',
    right_on='id',
    how='left'
)
merged

,langname,glottocode,word,count,id,year,simple_elaboration,regression_elaboration,dictsize_data,countryname,bila_language_name_mapping,mean,std,dictionary_count_mean
0,Turkish,nucl1301,fear,126.0,coo.31924067983704,1991.0,0.001036,1.952545,121674.0,Turkey,Turkish,2.948617,1.892269,14.502265
1,Turkish,nucl1301,pleasure,107.0,coo.31924067983704,1991.0,0.000879,3.153327,121674.0,Turkey,Turkish,3.083665,1.643463,14.502265
2,Turkish,nucl1301,regret,41.0,coo.31924067983704,1991.0,0.000337,2.670962,121674.0,Turkey,Turkish,1.768000,1.788567,14.502265
3,Turkish,nucl1301,admiration,10.0,coo.31924067983704,1991.0,0.000082,-0.854986,121674.0,Turkey,Turkish,2.027888,1.760460,14.502265
4,Turkish,nucl1301,anger,85.0,coo.31924067983704,1991.0,0.000699,0.579516,121674.0,Turkey,Turkish,3.460000,1.871473,14.502265
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
679,Japanese,nucl1643,disgust,13.0,uva.x004877953,1974.0,0.000046,-4.505457,282895.0,Japan,Japanese,2.854701,1.747655,33.718117
680,Japanese,nucl1643,frustration,6.0,uva.x004877953,1974.0,0.000021,-1.588076,282895.0,Japan,Japanese,3.111888,1.729330,33.718117
681,Japanese,nucl1643,loneliness,7.0,uva.x004877953,1974.0,0.000025,-1.873436,282895.0,Japan,Japanese,2.007764,1.817621,33.718117
682,Japanese,nucl1643,relief,89.0,uva.x004877953,1974.0,0.000315,3.907437,282895.0,Japan,Japanese,2.403263,1.417428,33.718117


In [991]:
merged = merged[merged['countryname'].notna()]

In [992]:
merged.columns

Index(['langname', 'glottocode', 'word', 'count', 'id', 'year',
       'simple_elaboration', 'regression_elaboration', 'dictsize_data',
       'countryname', 'bila_language_name_mapping', 'mean', 'std',
       'dictionary_count_mean'],
      dtype='object')

In [993]:
# clarify the meaning of mean
merged = merged.rename(columns={'mean': "response_mean"})
# Clarify what std we're talking about
merged = merged.rename(columns={'std': "response_std"})
# drop a bunch of columns

merged = merged[['langname', 'glottocode', 'word', 'count', 'id', 'year',
       'simple_elaboration', 'regression_elaboration', 'dictsize_data',
       'countryname', 'bila_language_name_mapping', 'response_mean',
       'response_std', 'dictionary_count_mean' ]]
merged = merged.sort_values(by=["langname", 'word'])
merged


,langname,glottocode,word,count,id,year,simple_elaboration,regression_elaboration,dictsize_data,countryname,bila_language_name_mapping,response_mean,response_std,dictionary_count_mean
237,Afrikaans,afri1274,admiration,6.0,mdp.39015054154474,1999.0,0.000028,-3.291632,212992.0,South Africa,Afrikaans,2.946429,1.712904,25.386412
238,Afrikaans,afri1274,anger,55.0,mdp.39015054154474,1999.0,0.000258,-6.741147,212992.0,South Africa,Afrikaans,2.785714,1.803043,25.386412
241,Afrikaans,afri1274,anxiety,30.0,mdp.39015054154474,1999.0,0.000141,-2.655449,212992.0,South Africa,Afrikaans,3.535714,1.765171,25.386412
251,Afrikaans,afri1274,boredom,5.0,mdp.39015054154474,1999.0,0.000023,-1.176312,212992.0,South Africa,Afrikaans,2.241071,1.889971,25.386412
239,Afrikaans,afri1274,compassion,15.0,mdp.39015054154474,1999.0,0.000070,-2.918472,212992.0,South Africa,Afrikaans,4.196429,1.505780,25.386412
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
384,Vietnamese,viet1252,love,308.0,uc1.31210024487306,2003.0,0.001143,-1.138291,269366.0,Vietnam,Vietnamese,4.786942,1.311475,32.105602
379,Vietnamese,viet1252,pleasure,216.0,uc1.31210024487306,2003.0,0.000802,3.511485,269366.0,Vietnam,Vietnamese,2.832192,1.885719,32.105602
380,Vietnamese,viet1252,regret,59.0,uc1.31210024487306,2003.0,0.000219,0.052866,269366.0,Vietnam,Vietnamese,1.534247,1.709743,32.105602
394,Vietnamese,viet1252,relief,74.0,uc1.31210024487306,2003.0,0.000275,2.404758,269366.0,Vietnam,Vietnamese,3.199313,1.720505,32.105602


In [994]:
merged = merged.reset_index(drop=True)
merged

,langname,glottocode,word,count,id,year,simple_elaboration,regression_elaboration,dictsize_data,countryname,bila_language_name_mapping,response_mean,response_std,dictionary_count_mean
0,Afrikaans,afri1274,admiration,6.0,mdp.39015054154474,1999.0,0.000028,-3.291632,212992.0,South Africa,Afrikaans,2.946429,1.712904,25.386412
1,Afrikaans,afri1274,anger,55.0,mdp.39015054154474,1999.0,0.000258,-6.741147,212992.0,South Africa,Afrikaans,2.785714,1.803043,25.386412
2,Afrikaans,afri1274,anxiety,30.0,mdp.39015054154474,1999.0,0.000141,-2.655449,212992.0,South Africa,Afrikaans,3.535714,1.765171,25.386412
3,Afrikaans,afri1274,boredom,5.0,mdp.39015054154474,1999.0,0.000023,-1.176312,212992.0,South Africa,Afrikaans,2.241071,1.889971,25.386412
4,Afrikaans,afri1274,compassion,15.0,mdp.39015054154474,1999.0,0.000070,-2.918472,212992.0,South Africa,Afrikaans,4.196429,1.505780,25.386412
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
679,Vietnamese,viet1252,love,308.0,uc1.31210024487306,2003.0,0.001143,-1.138291,269366.0,Vietnam,Vietnamese,4.786942,1.311475,32.105602
680,Vietnamese,viet1252,pleasure,216.0,uc1.31210024487306,2003.0,0.000802,3.511485,269366.0,Vietnam,Vietnamese,2.832192,1.885719,32.105602
681,Vietnamese,viet1252,regret,59.0,uc1.31210024487306,2003.0,0.000219,0.052866,269366.0,Vietnam,Vietnamese,1.534247,1.709743,32.105602
682,Vietnamese,viet1252,relief,74.0,uc1.31210024487306,2003.0,0.000275,2.404758,269366.0,Vietnam,Vietnamese,3.199313,1.720505,32.105602


In [997]:
print(merged['langname'].nunique(),
      merged['bila_language_name_mapping'].nunique(),
merged['word'].nunique(),
merged['countryname'].nunique(), sep="\n")

29
29
18
38


In [998]:
merged = merged.sort_values(by=["langname", 'countryname','word'])
merged = merged.reset_index(drop=True)
merged = merged.fillna(0)


In [999]:
merged

,langname,glottocode,word,count,id,year,simple_elaboration,regression_elaboration,dictsize_data,countryname,bila_language_name_mapping,response_mean,response_std,dictionary_count_mean
0,Afrikaans,afri1274,admiration,6.0,mdp.39015054154474,1999.0,0.000028,-3.291632,212992.0,South Africa,Afrikaans,2.946429,1.712904,25.386412
1,Afrikaans,afri1274,anger,55.0,mdp.39015054154474,1999.0,0.000258,-6.741147,212992.0,South Africa,Afrikaans,2.785714,1.803043,25.386412
2,Afrikaans,afri1274,anxiety,30.0,mdp.39015054154474,1999.0,0.000141,-2.655449,212992.0,South Africa,Afrikaans,3.535714,1.765171,25.386412
3,Afrikaans,afri1274,boredom,5.0,mdp.39015054154474,1999.0,0.000023,-1.176312,212992.0,South Africa,Afrikaans,2.241071,1.889971,25.386412
4,Afrikaans,afri1274,compassion,15.0,mdp.39015054154474,1999.0,0.000070,-2.918472,212992.0,South Africa,Afrikaans,4.196429,1.505780,25.386412
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
679,Vietnamese,viet1252,love,308.0,uc1.31210024487306,2003.0,0.001143,-1.138291,269366.0,Vietnam,Vietnamese,4.786942,1.311475,32.105602
680,Vietnamese,viet1252,pleasure,216.0,uc1.31210024487306,2003.0,0.000802,3.511485,269366.0,Vietnam,Vietnamese,2.832192,1.885719,32.105602
681,Vietnamese,viet1252,regret,59.0,uc1.31210024487306,2003.0,0.000219,0.052866,269366.0,Vietnam,Vietnamese,1.534247,1.709743,32.105602
682,Vietnamese,viet1252,relief,74.0,uc1.31210024487306,2003.0,0.000275,2.404758,269366.0,Vietnam,Vietnamese,3.199313,1.720505,32.105602


In [1000]:
merged.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 684 entries, 0 to 683
Data columns (total 14 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   langname                    684 non-null    object 
 1   glottocode                  684 non-null    object 
 2   word                        684 non-null    object 
 3   count                       684 non-null    float64
 4   id                          684 non-null    object 
 5   year                        684 non-null    float64
 6   simple_elaboration          684 non-null    float64
 7   regression_elaboration      684 non-null    float64
 8   dictsize_data               684 non-null    float64
 9   countryname                 684 non-null    object 
 10  bila_language_name_mapping  684 non-null    object 
 11  response_mean               684 non-null    float64
 12  response_std                684 non-null    float64
 13  dictionary_count_mean       684 non

In [1001]:
merged.to_csv(Path("~/Projects/hypocognition/data/processed/covid_bila_merge.csv").expanduser())

"Moving forward, could you create a table with the following columns (broken down into the 36 samples):
* The sample country
* The language
* The correlation between the log count and the response means
* The correlation between the log count and the response SDs
* The correlation between the log count and the absolute distance of the response means from the scale midpoint

In [1002]:
merged['response_mean'].mean()

np.float64(3.0094256137821955)

In [1003]:
import numpy as np

merged2 = merged.copy()

# log count
merged2["log_count"] = np.log(merged2["count"] + 1)

# absolute distance from midpoint
SCALE_MIDPOINT = merged2["response_mean"].mean()
merged2["abs_dist_midpoint"] = (
    merged2["response_mean"] - SCALE_MIDPOINT
).abs()

result = (
    merged2
    .groupby(["countryname", "langname", "id"])
    .agg(
        corr_logcount_response_mean=(
            "log_count",
            lambda x: x.corr(
                merged2.loc[x.index, "response_mean"]
            )
        ),
        corr_logcount_response_sd=(
            "log_count",
            lambda x: x.corr(
                merged2.loc[x.index, "response_std"]
            )
        ),
        corr_logcount_abs_dist_midpoint=(
            "log_count",
            lambda x: x.corr(
                merged2.loc[x.index, "abs_dist_midpoint"]
            )
        ),

    )
    .reset_index()
)


In [1004]:
result

,countryname,langname,id,corr_logcount_response_mean,corr_logcount_response_sd,corr_logcount_abs_dist_midpoint
0,Brazil,Brazilian Portuguese,txu.059173018640743,0.162436,-0.095865,0.058828
1,Bulgaria,Bulgarian,mdp.39015058560494,0.291458,-0.510940,0.265170
2,Chile,Latin American Spanish,mdp.39015050181174,0.371436,-0.362343,0.241683
3,China,Mandarin Chinese,uc1.l0088545033,0.191494,-0.404744,0.116165
4,Colombia,Latin American Spanish,mdp.39015050181174,0.236561,-0.175261,-0.137527
5,Croatia,Serbian-Croatian-Bosnian,mdp.39015012891506,-0.359706,-0.041463,-0.245080
6,Curacao,Dutch,uc1.31822004083036,-0.227172,-0.012792,-0.211599
7,Denmark,Danish,uc1.b3832576,-0.322866,-0.191735,0.313734
8,Egypt,Arabic,mdp.39015043036436,-0.114331,0.133291,-0.338184
9,Finland,Finnish,mdp.39015059174667,0.072036,-0.299668,-0.086978


In [1005]:
result.to_csv(Path("~/Projects/hypocognition/data/processed/corr_covid_bila.csv").expanduser())